# Sesión 2 · Notebook previo
## RAG sobre datos propios: embeddings, chunking y métricas de recuperación

**Curso:** Ciencia de datos aumentada con IA: del dato a la decisión
**Tiempo estimado:** 45 a 55 minutos · **Entrega:** este notebook ejecutado, antes de la clase

### Qué vas a hacer
1. Cargar los documentos de políticas de TiendaTech (el caso que usamos en la sesión 1) y partirlos en fragmentos (*chunks*) con dos estrategias distintas.
2. Generar embeddings por API y construir un buscador por similitud, sin librerías de base vectorial (para entender el mecanismo antes de usar una).
3. Medir la calidad de la recuperación con **hit rate** y **MRR** sobre 10 preguntas con respuesta conocida.
4. Comparar la búsqueda semántica (embeddings) contra una búsqueda clásica (TF-IDF / palabras clave).

En el taller de la sesión 2 usarán esta misma base para construir el RAG completo (recuperar + generar la respuesta) sobre la documentación de su propio caso.

## 0. Configuración

In [ ]:
!pip install -q google-genai scikit-learn pandas numpy requests

In [ ]:
import os, glob, re, json
import numpy as np
import pandas as pd
from google import genai
from google.genai import types

# Este notebook usa embeddings de Google AI Studio (Gemini), porque Anthropic no ofrece
# una API de embeddings. Si prefieres otro proveedor con API de embeddings, adapta la función
# `embeber` más abajo; el resto del notebook no cambia.
def obtener_clave(nombre):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre) or input(f"Pega tu {nombre}: ")

_cliente = genai.Client(api_key=obtener_clave("GEMINI_API_KEY"))
MODELO_EMBEDDING = "gemini-embedding-001"

def embeber(textos, tipo_tarea="RETRIEVAL_DOCUMENT"):
    """Recibe una lista de textos y devuelve una matriz (n_textos, dim) de embeddings."""
    out = []
    for i in range(0, len(textos), 20):  # en lotes, para no exceder límites de la API
        lote = textos[i:i+20]
        r = _cliente.models.embed_content(model=MODELO_EMBEDDING, contents=lote,
                                          config={"task_type": tipo_tarea})
        out.extend([e.values for e in r.embeddings])
    return np.array(out)

v = embeber(["prueba de conexión con la API de embeddings"])
print("Dimensión del embedding:", v.shape[1])

## 1. Cargar y trocear los documentos
Vas a comparar **dos estrategias de chunking**:

- **Chunking fijo:** cada fragmento tiene un número fijo de caracteres, con un pequeño solape (overlap) entre fragmentos consecutivos. Es simple pero puede partir una idea a la mitad.
- **Chunking por sección:** cada fragmento respeta los encabezados Markdown (`##`) del documento. Suele conservar mejor el contexto porque cada fragmento es una unidad de sentido completa.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
RUTA_DOCS = "drive/MyDrive/politicas_tiendatech"  # carpeta con los .md de políticas de TiendaTech
# Si corres este notebook en Colab, sube la carpeta 05_talleres/datos/politicas_tiendatech
# a /content/politicas_tiendatech antes de ejecutar esta celda.

archivos = sorted(glob.glob(f"{RUTA_DOCS}/*.md"))
documentos = {}
for ruta in archivos:
    with open(ruta, encoding="utf-8") as f:
        documentos[os.path.basename(ruta)] = f.read()
print(f"{len(documentos)} documentos cargados: {list(documentos)[:3]}...")

In [ ]:
def chunk_fijo(texto, tam=400, solape=60):
    chunks = []
    i = 0
    while i < len(texto):
        chunks.append(texto[i:i+tam])
        i += tam - solape
    return [c.strip() for c in chunks if c.strip()]

def chunk_por_seccion(texto):
    partes = re.split(r"(?m)^## ", texto)
    titulo_doc = partes[0].strip()
    chunks = []
    for parte in partes[1:]:
        chunks.append((titulo_doc + "\n\n## " + parte).strip())
    return chunks if chunks else [texto.strip()]

def construir_indice(estrategia):
    filas = []
    for nombre, texto in documentos.items():
        trozos = chunk_fijo(texto) if estrategia == "fijo" else chunk_por_seccion(texto)
        for j, t in enumerate(trozos):
            filas.append({"doc": nombre, "chunk_id": f"{nombre}#{j}", "texto": t})
    return pd.DataFrame(filas)

idx_fijo = construir_indice("fijo")
idx_seccion = construir_indice("seccion")
print(f"Chunking fijo: {len(idx_fijo)} fragmentos | Chunking por sección: {len(idx_seccion)} fragmentos")
idx_seccion.head(3)

## 2. Embeddings y búsqueda por similitud
Generamos el embedding de cada fragmento una sola vez (es la parte "cara"), y luego cada consulta se compara con todos los fragmentos usando **similitud coseno**. Con pocos documentos esto es perfectamente viable sin una base vectorial; en un caso con miles de documentos se usaría FAISS, Chroma o similar, pero el mecanismo de fondo es el mismo.

In [ ]:
def similitud_coseno(a, b):
    a = a / np.linalg.norm(a, axis=-1, keepdims=True)
    b = b / np.linalg.norm(b, axis=-1, keepdims=True)
    return a @ b.T

def indexar(df):
    df = df.copy()
    df["embedding"] = list(embeber(df["texto"].tolist(), tipo_tarea="RETRIEVAL_DOCUMENT"))
    return df

idx_fijo = indexar(idx_fijo)
idx_seccion = indexar(idx_seccion)
print("Índices listos.")

In [ ]:
def buscar(pregunta, df, k=3):
    v_pregunta = embeber([pregunta], tipo_tarea="RETRIEVAL_QUERY")[0]
    matriz = np.stack(df["embedding"].to_numpy())
    puntajes = similitud_coseno(v_pregunta[None, :], matriz)[0]
    top = np.argsort(-puntajes)[:k]
    return df.iloc[top].assign(puntaje=puntajes[top])[["doc", "chunk_id", "puntaje", "texto"]]

buscar("¿Cuánto tiempo tengo para devolver un producto sin dar motivo?", idx_seccion, k=3)

## 3. Conjunto de evaluación: 10 preguntas con documento de referencia
Para medir qué tan bien recupera cada índice, necesitamos saber **cuál es el documento correcto** para cada pregunta. Esto es exactamente lo mismo que hicimos en la sesión 1 con los tickets etiquetados: sin una referencia, solo podemos opinar "a ojo".

In [ ]:
PREGUNTAS_EVAL = [
    ("¿Cuánto tiempo tengo para devolver un producto sin dar motivo?", "politica_devoluciones_garantias.md"),
    ("Me cobraron dos veces el mismo pedido, ¿en cuánto tiempo me lo devuelven?", "politica_facturacion_pagos.md"),
    ("¿Cuánto cuesta el envío si compro menos de $200.000?", "politica_envios.md"),
    ("Mi pedido llegó golpeado, ¿qué debo hacer?", "politica_devoluciones_garantias.md"),
    ("¿En cuánto tiempo responden un caso de soporte técnico urgente?", "faq_soporte_tecnico.md"),
    ("¿Cuál es la garantía de la batería del portátil Lenovo IdeaPad?", "manual_portatil_lenovo_ideapad.md"),
    ("Mi lavadora LG vibra mucho, ¿qué puede ser?", "manual_lavadora_lg_wm3900.md"),
    ("¿Cómo solicito que eliminen mis datos personales?", "politica_privacidad_datos.md"),
    ("¿Cuándo se activa la atención prioritaria para un cliente?", "politica_atencion_prioritaria.md"),
    ("¿Los audífonos Sony tienen garantía si el cable llega dañado?", "manual_audifonos_sony_wh1000xm5.md"),
]
print(f"{len(PREGUNTAS_EVAL)} preguntas de evaluación")

## 4. Métricas: hit rate y MRR
- **Hit rate@k:** proporción de preguntas para las que el documento correcto aparece entre los k fragmentos recuperados.
- **MRR (Mean Reciprocal Rank):** promedio de 1 dividido por la posición del primer fragmento correcto (1 si acierta de primero, 0.5 si es el segundo, 0 si no aparece en el top k). Premia no solo acertar, sino acertar de primero.

In [ ]:
def evaluar_indice(nombre, df, k=3):
    filas = []
    for pregunta, doc_correcto in PREGUNTAS_EVAL:
        resultados = buscar(pregunta, df, k=k)
        docs_recuperados = resultados["doc"].tolist()
        acierto = doc_correcto in docs_recuperados
        rango_reciproco = 0.0
        if acierto:
            posicion = docs_recuperados.index(doc_correcto) + 1
            rango_reciproco = 1 / posicion
        filas.append({"pregunta": pregunta[:45] + "...", "doc_correcto": doc_correcto,
                      "recuperados": docs_recuperados, "acierto": acierto, "rango_reciproco": rango_reciproco})
    df_res = pd.DataFrame(filas)
    resumen = {"indice": nombre, "hit_rate@%d" % k: df_res["acierto"].mean(), "mrr": df_res["rango_reciproco"].mean()}
    return df_res, resumen

RESULTADOS_CHUNKING = {}
df_r1, res1 = evaluar_indice("Chunking fijo (embeddings)", idx_fijo)
df_r2, res2 = evaluar_indice("Chunking por sección (embeddings)", idx_seccion)
RESULTADOS_CHUNKING[res1["indice"]] = res1
RESULTADOS_CHUNKING[res2["indice"]] = res2
pd.DataFrame(RESULTADOS_CHUNKING).T

### TU TURNO 1
Mira `df_r1` y `df_r2` (agrega una celda si quieres imprimirlos completos) y responde:
1. ¿Qué estrategia de chunking recuperó mejor y en qué tipo de pregunta se nota más la diferencia?
2. Elige una pregunta donde falle el chunking fijo pero acierte el de por sección (o viceversa) y explica por qué, mirando el texto del fragmento recuperado.

In [ ]:
df_r1  # o df_r2 — inspecciona los casos donde 'acierto' es False

respuesta_1 = """
Escribe aquí tu respuesta.
"""

## 5. Comparación con una búsqueda clásica (TF-IDF)
Antes de usar embeddings, la forma estándar de buscar por texto era por **coincidencia de palabras** (TF-IDF + similitud coseno). No necesita API ni GPU, y en preguntas donde la pregunta comparte palabras exactas con el documento puede funcionar sorprendentemente bien. Su punto débil es que no entiende sinónimos ni paráfrasis.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def indexar_tfidf(df):
    vectorizador = TfidfVectorizer()
    matriz = vectorizador.fit_transform(df["texto"])
    return vectorizador, matriz

def buscar_tfidf(pregunta, df, vectorizador, matriz, k=3):
    v_pregunta = vectorizador.transform([pregunta])
    puntajes = cosine_similarity(v_pregunta, matriz)[0]
    top = np.argsort(-puntajes)[:k]
    return df.iloc[top].assign(puntaje=puntajes[top])[["doc", "chunk_id", "puntaje"]]

def evaluar_tfidf(nombre, df, k=3):
    vectorizador, matriz = indexar_tfidf(df)
    filas = []
    for pregunta, doc_correcto in PREGUNTAS_EVAL:
        recuperados = buscar_tfidf(pregunta, df, vectorizador, matriz, k=k)["doc"].tolist()
        acierto = doc_correcto in recuperados
        rr = 1 / (recuperados.index(doc_correcto) + 1) if acierto else 0.0
        filas.append({"acierto": acierto, "rango_reciproco": rr})
    df_res = pd.DataFrame(filas)
    return {"indice": nombre, "hit_rate@%d" % k: df_res["acierto"].mean(), "mrr": df_res["rango_reciproco"].mean()}

res_tfidf = evaluar_tfidf("Chunking por sección (TF-IDF)", idx_seccion)
RESULTADOS_CHUNKING[res_tfidf["indice"]] = res_tfidf
pd.DataFrame(RESULTADOS_CHUNKING).T

### TU TURNO 2
1. ¿En qué se diferencia el resultado de TF-IDF frente a los embeddings? ¿Es mejor, peor o depende de la pregunta?
2. Escribe **una pregunta nueva que use sinónimos o una forma distinta de preguntar** (por ejemplo, sin repetir las palabras exactas del documento) y compara cómo le va a TF-IDF frente a embeddings para esa pregunta específica.

In [ ]:
pregunta_nueva = "..."  # TU TURNO: escribe tu pregunta con sinónimos

print("--- Embeddings ---")
print(buscar(pregunta_nueva, idx_seccion, k=3)[["doc", "puntaje"]])
print("\n--- TF-IDF ---")
vec, mat = indexar_tfidf(idx_seccion)
print(buscar_tfidf(pregunta_nueva, idx_seccion, vec, mat, k=3))

respuesta_2 = """
Escribe aquí tu respuesta.
"""

## 6. (Preparación para el taller) Generar la respuesta con el contexto recuperado
En el taller construirán el RAG completo: recuperar los fragmentos y pasárselos al LLM para que responda **solo con base en ellos**. Aquí dejamos la función lista para usar en clase.

In [ ]:

from google import genai
from google.genai import types

# Cliente de Gemini
_cliente_llm = genai.Client(
    api_key=obtener_clave("GEMINI_API_KEY")
)

def responder_con_rag(pregunta, df_indice, k=3):
    fragmentos = buscar(pregunta, df_indice, k=k)

    contexto = "\n\n---\n\n".join(
        f"[Fuente: {r.doc}]\n{r.texto}"
        for r in fragmentos.itertuples()
    )

    prompt = f"""Responde la pregunta del cliente usando ÚNICAMENTE la información del CONTEXTO.
Si la respuesta no está en el contexto, di exactamente: "No tengo esa información en las políticas disponibles."
Cita el documento fuente entre paréntesis al final de la respuesta.

CONTEXTO:
{contexto}

PREGUNTA: {pregunta}

RESPUESTA:"""

    r = _cliente_llm.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            max_output_tokens=300,
            temperature=0
        )
    )

    return r.text, fragmentos


respuesta, fuentes = responder_con_rag(
    "¿Cuánto tiempo tengo para devolver un producto sin dar motivo?",
    idx_seccion
)

print(respuesta)


## 7. Cierre
### TU TURNO 3 · Reflexión final (máx. 5 líneas)
¿Qué estrategia de chunking y qué método de búsqueda usarías para la documentación de tu propio proyecto, y por qué? Ten en cuenta el tamaño de tus documentos y qué tan literal o parafraseado esperas que sea el lenguaje de las preguntas.

In [ ]:
respuesta_3 = """
Escribe aquí tu reflexión.
"""

DECLARACION_IAG = """
Nivel de uso de IAG: (1 a 5, según el acuerdo del curso)
Herramientas usadas y para qué:
Cómo verifiqué la calidad y veracidad del contenido generado:
"""

### Antes de entregar
- Sube la carpeta `politicas_tiendatech/` a tu sesión de Colab (o móntala desde Drive) antes de ejecutar.
- Ejecuta **Entorno de ejecución → Reiniciar y ejecutar todo** y verifica que no haya errores.
- Comprueba que las celdas `respuesta_1` a `respuesta_3` y `DECLARACION_IAG` estén completas.
- **No compartas tu clave de API.**